# 02. Model Training

UCI Hydraulic System 데이터를 이용하여
냉각기, 밸브, 펌프, 축압기 상태 예측 모델을 학습하고 검증 방식을 비교한다.

### 목표

1. 전처리된 특징 데이터 불러오기
2. Train / Validation / Test 데이터 구성
3. RandomForest 기준 모델 학습
4. 10초 / 20초 / 30초 / 60초 조기판별 모델 확인
5. RandomForest / LightGBM 비교
6. Stratified 분할과 시간순서 검증 방식 확인
7. 추가 모델 비교에 사용할 학습 데이터 준비


In [1]:
import pandas as pd

from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix
)

from lightgbm import LGBMClassifier


## 2. 특징 데이터 준비

### 2-1. 특징 데이터 파일 확인

In [2]:
processed_dir = Path("../data/processed")

feature_files = [
    "features_10s.parquet",
    "features_20s.parquet",
    "features_30s.parquet",
    "features_60s.parquet"
]

for file_name in feature_files:
    file_path = processed_dir / file_name
    print(file_name, "→", file_path.exists())

features_10s.parquet → True
features_20s.parquet → True
features_30s.parquet → True
features_60s.parquet → True


### 2-2. 60초 특징 데이터 불러오기

In [3]:
features_60s = pd.read_parquet(
    processed_dir / "features_60s.parquet"
)

print("60초 특징 데이터 불러오기 완료")

60초 특징 데이터 불러오기 완료


### 2-3. 데이터 크기 확인

In [4]:
print("데이터 크기:", features_60s.shape)
print("행 개수:", features_60s.shape[0])
print("열 개수:", features_60s.shape[1])

데이터 크기: (2205, 120)
행 개수: 2205
열 개수: 120


### 2-4. 컬럼 확인

In [5]:
print("전체 컬럼 수:", len(features_60s.columns))

for col in features_60s.columns:
    print(col)

전체 컬럼 수: 120
cycle_id
PS1_mean
PS1_std
PS1_min
PS1_max
PS1_range
PS1_slope
PS1_rms
PS2_mean
PS2_std
PS2_min
PS2_max
PS2_range
PS2_slope
PS2_rms
PS3_mean
PS3_std
PS3_min
PS3_max
PS3_range
PS3_slope
PS3_rms
PS4_mean
PS4_std
PS4_min
PS4_max
PS4_range
PS4_slope
PS4_rms
PS5_mean
PS5_std
PS5_min
PS5_max
PS5_range
PS5_slope
PS5_rms
PS6_mean
PS6_std
PS6_min
PS6_max
PS6_range
PS6_slope
PS6_rms
EPS1_mean
EPS1_std
EPS1_min
EPS1_max
EPS1_range
EPS1_slope
EPS1_rms
FS1_mean
FS1_std
FS1_min
FS1_max
FS1_range
FS1_slope
FS1_rms
FS2_mean
FS2_std
FS2_min
FS2_max
FS2_range
FS2_slope
FS2_rms
TS1_mean
TS1_std
TS1_min
TS1_max
TS1_range
TS1_slope
TS1_rms
TS2_mean
TS2_std
TS2_min
TS2_max
TS2_range
TS2_slope
TS2_rms
TS3_mean
TS3_std
TS3_min
TS3_max
TS3_range
TS3_slope
TS3_rms
TS4_mean
TS4_std
TS4_min
TS4_max
TS4_range
TS4_slope
TS4_rms
VS1_mean
VS1_std
VS1_min
VS1_max
VS1_range
VS1_slope
VS1_rms
CE_mean
CE_std
CE_min
CE_max
CE_range
CE_slope
CE_rms
CP_mean
CP_std
CP_min
CP_max
CP_range
CP_slope
CP_rms
SE_mean
S

### 2-5. 결측값 확인

In [6]:
missing_values = features_60s.isnull().sum()

print("전체 결측값 수:", missing_values.sum())
print()
print("결측값이 있는 컬럼:")

print(missing_values[missing_values > 0])

전체 결측값 수: 0

결측값이 있는 컬럼:
Series([], dtype: int64)


## 3. 학습 데이터 구성

#### 3-1-1. profile.txt 실제 경로 찾기

In [7]:
# 정답 라벨이 들어있는 profile.txt의 실제 위치를 확인
print("현재 Jupyter 위치:", Path.cwd())

search_root = Path.cwd().parent

profile_files = list(search_root.rglob("profile.txt"))

print("\n찾은 profile.txt:")
for path in profile_files:
    print(path)

현재 Jupyter 위치: C:\ai-first-project\notebooks

찾은 profile.txt:
C:\ai-first-project\data\raw\uci_hydraulic\extracted\profile.txt


### 3-1. 입력 특징(X)과 정답(y) 구분

In [8]:
# 60초 특징 데이터
X_all = features_60s.copy()

# profile.txt 경로
profile_path = Path("../data/raw/uci_hydraulic/extracted/profile.txt")

# 정답 데이터 불러오기
profile = pd.read_csv(
    profile_path,
    sep=r"\s+",
    header=None
)

# 정답 컬럼 이름 지정
profile.columns = [
    "cooler",
    "valve",
    "pump",
    "accumulator",
    "stable_flag"
]

# cycle_id 추가
profile.insert(0, "cycle_id", range(1, len(profile) + 1))

print("특징 데이터 크기:", X_all.shape)
print("정답 데이터 크기:", profile.shape)

display(profile.head())

특징 데이터 크기: (2205, 120)
정답 데이터 크기: (2205, 6)


,cycle_id,cooler,valve,pump,accumulator,stable_flag
0,1,3,100,0,130,1
1,2,3,100,0,130,1
2,3,3,100,0,130,1
3,4,3,100,0,130,1
4,5,3,100,0,130,1


### 3-2. 냉각기 라벨 확인

* 3   → 고장 근접
* 20  → 효율 저하
* 100 → 정상

In [9]:
cooler_counts = profile["cooler"].value_counts().sort_index()

print("냉각기 라벨 종류:")
print(sorted(profile["cooler"].unique()))

print("\n냉각기 라벨별 개수:")
print(cooler_counts)

냉각기 라벨 종류:
[np.int64(3), np.int64(20), np.int64(100)]

냉각기 라벨별 개수:
cooler
3      732
20     732
100    741
Name: count, dtype: int64


### 3-3. 밸브 라벨 확인

* 73  → 고장 근접
* 80  → 심각 지연
* 90  → 미세 지연
* 100 → 정상

In [10]:
valve_counts = profile["valve"].value_counts().sort_index()

print("밸브 라벨 종류:")
print(sorted(profile["valve"].unique()))

print("\n밸브 라벨별 개수:")
print(valve_counts)

밸브 라벨 종류:
[np.int64(73), np.int64(80), np.int64(90), np.int64(100)]

밸브 라벨별 개수:
valve
73      360
80      360
90      360
100    1125
Name: count, dtype: int64


### 3-4. 펌프 라벨 확인

* 0 → 누설 없음
* 1 → 약한 누설
* 2 → 심각한 누설

In [11]:
pump_counts = profile["pump"].value_counts().sort_index()

print("펌프 라벨 종류:")
print(sorted(profile["pump"].unique()))

print("\n펌프 라벨별 개수:")
print(pump_counts)

펌프 라벨 종류:
[np.int64(0), np.int64(1), np.int64(2)]

펌프 라벨별 개수:
pump
0    1221
1     492
2     492
Name: count, dtype: int64


### 3-5. 축압기 라벨 확인

* 90  → 고장 근접
* 100 → 심각 저하
* 115 → 약한 저하
* 130 → 정상

In [12]:
accumulator_counts = profile["accumulator"].value_counts().sort_index()

print("축압기 라벨 종류:")
print(sorted(profile["accumulator"].unique()))

print("\n축압기 라벨별 개수:")
print(accumulator_counts)

축압기 라벨 종류:
[np.int64(90), np.int64(100), np.int64(115), np.int64(130)]

축압기 라벨별 개수:
accumulator
90     808
100    399
115    399
130    599
Name: count, dtype: int64


### 3-6. 기본 시간순 Train / Validation / Test 분할


In [13]:
# cycle_id 기준 데이터 분할
train_ids = range(1, 1544)
val_ids = range(1544, 1875)
test_ids = range(1875, 2206)

train_data = features_60s[features_60s["cycle_id"].isin(train_ids)]
val_data = features_60s[features_60s["cycle_id"].isin(val_ids)]
test_data = features_60s[features_60s["cycle_id"].isin(test_ids)]

print("Train 개수:", len(train_data))
print("Validation 개수:", len(val_data))
print("Test 개수:", len(test_data))

print("\nTrain cycle_id 범위:",
      train_data["cycle_id"].min(), "~", train_data["cycle_id"].max())

print("Validation cycle_id 범위:",
      val_data["cycle_id"].min(), "~", val_data["cycle_id"].max())

print("Test cycle_id 범위:",
      test_data["cycle_id"].min(), "~", test_data["cycle_id"].max())

Train 개수: 1543
Validation 개수: 331
Test 개수: 331

Train cycle_id 범위: 1 ~ 1543
Validation cycle_id 범위: 1544 ~ 1874
Test cycle_id 범위: 1875 ~ 2205


### 3-7. Train / Validation / Test 중복 확인


In [14]:
train_id_set = set(train_data["cycle_id"])
val_id_set = set(val_data["cycle_id"])
test_id_set = set(test_data["cycle_id"])

train_val_overlap = train_id_set & val_id_set
train_test_overlap = train_id_set & test_id_set
val_test_overlap = val_id_set & test_id_set

print("Train ↔ Validation 중복:", len(train_val_overlap))
print("Train ↔ Test 중복:", len(train_test_overlap))
print("Validation ↔ Test 중복:", len(val_test_overlap))

all_ids = train_id_set | val_id_set | test_id_set
print("\n전체 cycle_id 개수:", len(all_ids))

Train ↔ Validation 중복: 0
Train ↔ Test 중복: 0
Validation ↔ Test 중복: 0

전체 cycle_id 개수: 2205


## 4. 60초 RandomForest 기준 모델

60초 전체 특징 데이터를 사용하여 RandomForest를 기준 모델(Baseline)로 학습한다.

RandomForest는 별도의 스케일링 없이 통계 특징을 사용할 수 있고 다중분류에 적용하기 쉬워 기준 모델로 사용한다. 이후 10초 / 20초 / 30초 조기판별 모델과 LightGBM의 성능을 동일한 평가 지표로 비교한다.

### 4-1. 냉각기 모델 학습


In [15]:
# 모델 입력 특징 컬럼
feature_cols = [
    col for col in features_60s.columns
    if col != "cycle_id"
]

# Train / Validation 입력 데이터
X_train = train_data[feature_cols]
X_val = val_data[feature_cols]

# cycle_id에 맞춰 정답 데이터 가져오기
y_train_cooler = profile[
    profile["cycle_id"].isin(train_ids)
]["cooler"]

y_val_cooler = profile[
    profile["cycle_id"].isin(val_ids)
]["cooler"]

print("X_train:", X_train.shape)
print("y_train:", y_train_cooler.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val_cooler.shape)

X_train: (1543, 119)
y_train: (1543,)
X_val: (331, 119)
y_val: (331,)


In [16]:
rf_cooler = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_cooler.fit(X_train, y_train_cooler)

print("냉각기 RandomForest 학습 완료")

냉각기 RandomForest 학습 완료


### 4-2. 밸브 모델 학습

In [17]:
y_train_valve = profile[
    profile["cycle_id"].isin(train_ids)
]["valve"]

y_val_valve = profile[
    profile["cycle_id"].isin(val_ids)
]["valve"]

rf_valve = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_valve.fit(X_train, y_train_valve)

print("밸브 RandomForest 학습 완료")

밸브 RandomForest 학습 완료


### 4-3. 펌프 모델 학습

In [18]:
y_train_pump = profile[
    profile["cycle_id"].isin(train_ids)
]["pump"]

y_val_pump = profile[
    profile["cycle_id"].isin(val_ids)
]["pump"]

rf_pump = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_pump.fit(X_train, y_train_pump)

print("펌프 RandomForest 학습 완료")

펌프 RandomForest 학습 완료


### 4-4. 축압기 모델 학습

In [19]:
y_train_accumulator = profile[
    profile["cycle_id"].isin(train_ids)
]["accumulator"]

y_val_accumulator = profile[
    profile["cycle_id"].isin(val_ids)
]["accumulator"]

rf_accumulator = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_accumulator.fit(X_train, y_train_accumulator)

print("축압기 RandomForest 학습 완료")

축압기 RandomForest 학습 완료


### 4-5. Accuracy / Macro F1 평가

In [20]:
# Validation 데이터 예측
pred_cooler = rf_cooler.predict(X_val)
pred_valve = rf_valve.predict(X_val)
pred_pump = rf_pump.predict(X_val)
pred_accumulator = rf_accumulator.predict(X_val)

# 평가 결과 저장
results = pd.DataFrame({
    "component": ["cooler", "valve", "pump", "accumulator"],
    "accuracy": [
        accuracy_score(y_val_cooler, pred_cooler),
        accuracy_score(y_val_valve, pred_valve),
        accuracy_score(y_val_pump, pred_pump),
        accuracy_score(y_val_accumulator, pred_accumulator)
    ],
    "macro_f1": [
        f1_score(y_val_cooler, pred_cooler, average="macro"),
        f1_score(y_val_valve, pred_valve, average="macro"),
        f1_score(y_val_pump, pred_pump, average="macro"),
        f1_score(y_val_accumulator, pred_accumulator, average="macro")
    ]
})

display(results)

,component,accuracy,macro_f1
0,cooler,1.000000,1.000000
1,valve,0.561934,0.179884
2,pump,0.794562,0.715191
3,accumulator,0.365559,0.178466


### 4-5-1. Train / Validation 라벨 분포 확인

In [21]:
components = ["cooler", "valve", "pump", "accumulator"]

for component in components:
    print("=" * 50)
    print(component.upper())

    print("\nTrain:")
    print(
        profile[
            profile["cycle_id"].isin(train_ids)
        ][component].value_counts().sort_index()
    )

    print("\nValidation:")
    print(
        profile[
            profile["cycle_id"].isin(val_ids)
        ][component].value_counts().sort_index()
    )

    print()

COOLER

Train:
cooler
3      732
20     732
100     79
Name: count, dtype: int64

Validation:
cooler
100    331
Name: count, dtype: int64

VALVE

Train:
valve
73     240
80     240
90     240
100    823
Name: count, dtype: int64

Validation:
valve
73      50
80      50
90      45
100    186
Name: count, dtype: int64

PUMP

Train:
pump
0    887
1    328
2    328
Name: count, dtype: int64

Validation:
pump
0    182
1     67
2     82
Name: count, dtype: int64

ACCUMULATOR

Train:
accumulator
90     545
100    266
115    266
130    466
Name: count, dtype: int64

Validation:
accumulator
90     121
115     77
130    133
Name: count, dtype: int64



### 4-6. Confusion Matrix 확인

In [22]:
components_info = [
    ("Cooler", "cooler", y_val_cooler, pred_cooler),
    ("Valve", "valve", y_val_valve, pred_valve),
    ("Pump", "pump", y_val_pump, pred_pump),
    ("Accumulator", "accumulator", y_val_accumulator, pred_accumulator)
]

for name, column, y_true, y_pred in components_info:
    # Validation에 없는 상태도 포함해서 전체 라벨 사용
    labels = sorted(profile[column].unique())

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=labels
    )

    print("=" * 50)
    print(name)
    print("Labels:", labels)
    print(cm)
    print()

Cooler
Labels: [np.int64(3), np.int64(20), np.int64(100)]
[[  0   0   0]
 [  0   0   0]
 [  0   0 331]]

Valve
Labels: [np.int64(73), np.int64(80), np.int64(90), np.int64(100)]
[[  0   0   0  50]
 [  0   0   0  50]
 [  0   0   0  45]
 [  0   0   0 186]]

Pump
Labels: [np.int64(0), np.int64(1), np.int64(2)]
[[182   0   0]
 [ 34  33   0]
 [  0  34  48]]

Accumulator
Labels: [np.int64(90), np.int64(100), np.int64(115), np.int64(130)]
[[121   0   0   0]
 [  0   0   0   0]
 [ 77   0   0   0]
 [133   0   0   0]]



## 5. 조기판별 모델

### 5-1. 10초 RandomForest 모델 학습 및 평가

In [23]:
# 10초 데이터 준비

# 10초 특징 데이터 불러오기
features_10s = pd.read_parquet(
    processed_dir / "features_10s.parquet"
)

# Train / Validation 분할
train_10s = features_10s[
    features_10s["cycle_id"].isin(train_ids)
]

val_10s = features_10s[
    features_10s["cycle_id"].isin(val_ids)
]

# cycle_id를 제외한 센서 특징
feature_cols_10s = [
    col for col in features_10s.columns
    if col != "cycle_id"
]

X_train_10s = train_10s[feature_cols_10s]
X_val_10s = val_10s[feature_cols_10s]

print("10초 전체 데이터:", features_10s.shape)
print("Train:", X_train_10s.shape)
print("Validation:", X_val_10s.shape)

10초 전체 데이터: (2205, 120)
Train: (1543, 119)
Validation: (331, 119)


In [24]:
# 네 부품 모델 학습

rf_10s_models = {}

for component in ["cooler", "valve", "pump", "accumulator"]:

    y_train = profile[
        profile["cycle_id"].isin(train_ids)
    ][component]

    model = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_10s, y_train)

    rf_10s_models[component] = model

    print(component, "10초 모델 학습 완료")

cooler 10초 모델 학습 완료
valve 10초 모델 학습 완료
pump 10초 모델 학습 완료
accumulator 10초 모델 학습 완료


In [25]:
# 10초 성능 평가

results_10s = []

for component in ["cooler", "valve", "pump", "accumulator"]:

    y_val = profile[
        profile["cycle_id"].isin(val_ids)
    ][component]

    pred = rf_10s_models[component].predict(X_val_10s)

    results_10s.append({
        "component": component,
        "accuracy": accuracy_score(y_val, pred),
        "macro_f1": f1_score(
            y_val,
            pred,
            average="macro"
        )
    })

results_10s = pd.DataFrame(results_10s)

display(results_10s)

,component,accuracy,macro_f1
0,cooler,1.000000,1.000000
1,valve,0.996979,0.996521
2,pump,0.731118,0.510491
3,accumulator,0.392749,0.224345


### 5-1-1. 10초 / 60초 특징 구조 확인

In [26]:
# 밸브가 60초에서는 거의 못 맞췄는데 10초에서는 거의 100%**라서 변화가 너무 커

cols_10s = features_10s.columns.tolist()
cols_60s = features_60s.columns.tolist()

print("10초 데이터 크기:", features_10s.shape)
print("60초 데이터 크기:", features_60s.shape)

print("\n컬럼 개수 동일:", len(cols_10s) == len(cols_60s))
print("컬럼 이름/순서 동일:", cols_10s == cols_60s)

print("\n10초에만 있는 컬럼:")
print(sorted(set(cols_10s) - set(cols_60s)))

print("\n60초에만 있는 컬럼:")
print(sorted(set(cols_60s) - set(cols_10s)))

10초 데이터 크기: (2205, 120)
60초 데이터 크기: (2205, 120)

컬럼 개수 동일: True
컬럼 이름/순서 동일: True

10초에만 있는 컬럼:
[]

60초에만 있는 컬럼:
[]


### 5-2. 20초 RandomForest 모델 학습 및 평가

In [27]:
# 20초 데이터 준비

features_20s = pd.read_parquet(
    processed_dir / "features_20s.parquet"
)

train_20s = features_20s[
    features_20s["cycle_id"].isin(train_ids)
]

val_20s = features_20s[
    features_20s["cycle_id"].isin(val_ids)
]

feature_cols_20s = [
    col for col in features_20s.columns
    if col != "cycle_id"
]

X_train_20s = train_20s[feature_cols_20s]
X_val_20s = val_20s[feature_cols_20s]

print("20초 전체 데이터:", features_20s.shape)
print("Train:", X_train_20s.shape)
print("Validation:", X_val_20s.shape)

20초 전체 데이터: (2205, 120)
Train: (1543, 119)
Validation: (331, 119)


In [28]:
# 4개 부품 학습

rf_20s_models = {}

for component in ["cooler", "valve", "pump", "accumulator"]:

    y_train = profile[
        profile["cycle_id"].isin(train_ids)
    ][component]

    model = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_20s, y_train)

    rf_20s_models[component] = model

    print(component, "20초 모델 학습 완료")

cooler 20초 모델 학습 완료
valve 20초 모델 학습 완료
pump 20초 모델 학습 완료
accumulator 20초 모델 학습 완료


In [29]:
# 20초 성능 평가

results_20s = []

for component in ["cooler", "valve", "pump", "accumulator"]:

    y_val = profile[
        profile["cycle_id"].isin(val_ids)
    ][component]

    pred = rf_20s_models[component].predict(X_val_20s)

    results_20s.append({
        "component": component,
        "accuracy": accuracy_score(y_val, pred),
        "macro_f1": f1_score(
            y_val,
            pred,
            average="macro"
        )
    })

results_20s = pd.DataFrame(results_20s)

display(results_20s)

,component,accuracy,macro_f1
0,cooler,1.000000,1.000000
1,valve,0.561934,0.179884
2,pump,0.885196,0.833098
3,accumulator,0.365559,0.178466


### 5-3. 30초 RandomForest 모델 학습 및 평가

In [30]:
# 30초 데이터 준비

features_30s = pd.read_parquet(
    processed_dir / "features_30s.parquet"
)

train_30s = features_30s[
    features_30s["cycle_id"].isin(train_ids)
]

val_30s = features_30s[
    features_30s["cycle_id"].isin(val_ids)
]

feature_cols_30s = [
    col for col in features_30s.columns
    if col != "cycle_id"
]

X_train_30s = train_30s[feature_cols_30s]
X_val_30s = val_30s[feature_cols_30s]

print("30초 전체 데이터:", features_30s.shape)
print("Train:", X_train_30s.shape)
print("Validation:", X_val_30s.shape)

30초 전체 데이터: (2205, 120)
Train: (1543, 119)
Validation: (331, 119)


In [31]:
# 4개 부품 학습

rf_30s_models = {}

for component in ["cooler", "valve", "pump", "accumulator"]:

    y_train = profile[
        profile["cycle_id"].isin(train_ids)
    ][component]

    model = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_30s, y_train)

    rf_30s_models[component] = model

    print(component, "30초 모델 학습 완료")

cooler 30초 모델 학습 완료
valve 30초 모델 학습 완료
pump 30초 모델 학습 완료
accumulator 30초 모델 학습 완료


In [32]:
# 30초 성능 평가

results_30s = []

for component in ["cooler", "valve", "pump", "accumulator"]:

    y_val = profile[
        profile["cycle_id"].isin(val_ids)
    ][component]

    pred = rf_30s_models[component].predict(X_val_30s)

    results_30s.append({
        "component": component,
        "accuracy": accuracy_score(y_val, pred),
        "macro_f1": f1_score(
            y_val,
            pred,
            average="macro"
        )
    })

results_30s = pd.DataFrame(results_30s)

display(results_30s)

,component,accuracy,macro_f1
0,cooler,1.000000,1.000000
1,valve,0.561934,0.179884
2,pump,0.854985,0.775802
3,accumulator,0.365559,0.178466


### 5-4. 10초 / 20초 / 30초 / 60초 성능 비교

In [33]:
# 각 결과에 시간 정보 추가
compare_10s = results_10s.copy()
compare_10s["window"] = "10s"

compare_20s = results_20s.copy()
compare_20s["window"] = "20s"

compare_30s = results_30s.copy()
compare_30s["window"] = "30s"

compare_60s = results.copy()
compare_60s["window"] = "60s"

# 하나의 표로 합치기
comparison_results = pd.concat(
    [
        compare_10s,
        compare_20s,
        compare_30s,
        compare_60s
    ],
    ignore_index=True
)

# 보기 편하게 컬럼 순서 변경
comparison_results = comparison_results[
    ["window", "component", "accuracy", "macro_f1"]
]

display(comparison_results)

,window,component,accuracy,macro_f1
0,10s,cooler,1.000000,1.000000
1,10s,valve,0.996979,0.996521
2,10s,pump,0.731118,0.510491
3,10s,accumulator,0.392749,0.224345
4,20s,cooler,1.000000,1.000000
5,20s,valve,0.561934,0.179884
6,20s,pump,0.885196,0.833098
7,20s,accumulator,0.365559,0.178466
8,30s,cooler,1.000000,1.000000
9,30s,valve,0.561934,0.179884


In [34]:
macro_f1_table = comparison_results.pivot(
    index="component",
    columns="window",
    values="macro_f1"
)

# 부품 순서 지정
component_order = [
    "cooler",
    "valve",
    "pump",
    "accumulator"
]

# 시간 순서 지정
macro_f1_table = macro_f1_table.loc[
    component_order,
    ["10s", "20s", "30s", "60s"]
]

display(macro_f1_table)

window,10s,20s,30s,60s
component,,,,
cooler,1.000000,1.000000,1.000000,1.000000
valve,0.996521,0.179884,0.179884,0.179884
pump,0.510491,0.833098,0.775802,0.715191
accumulator,0.224345,0.178466,0.178466,0.178466


### 5-5. 시간 구간별 최고 성능 확인

In [35]:
best_window_results = []

for component in component_order:
    scores = macro_f1_table.loc[component]

    best_window = scores.idxmax()
    best_score = scores.max()

    best_window_results.append({
        "component": component,
        "best_window": best_window,
        "best_macro_f1": best_score
    })

best_window_results = pd.DataFrame(best_window_results)

display(best_window_results)

,component,best_window,best_macro_f1
0,cooler,10s,1.000000
1,valve,10s,0.996521
2,pump,20s,0.833098
3,accumulator,10s,0.224345


In [36]:
# 축압기 Validation 정답
y_val_acc_10s = profile[
    profile["cycle_id"].isin(val_ids)
]["accumulator"]

# 축압기 10초 예측
pred_acc_10s = rf_10s_models["accumulator"].predict(X_val_10s)

# 전체 축압기 라벨
acc_labels = sorted(profile["accumulator"].unique())

cm_acc_10s = confusion_matrix(
    y_val_acc_10s,
    pred_acc_10s,
    labels=acc_labels
)

print("축압기 라벨:", acc_labels)
print()
print(cm_acc_10s)

축압기 라벨: [np.int64(90), np.int64(100), np.int64(115), np.int64(130)]

[[121   0   0   0]
 [  0   0   0   0]
 [ 77   0   0   0]
 [124   0   0   9]]


## 6. LightGBM 비교

### 6-1. LightGBM 설치 여부 확인

In [37]:
try:
    import lightgbm as lgb
    print("LightGBM 설치됨")
    print("버전:", lgb.__version__)

except ImportError:
    print("LightGBM이 설치되어 있지 않습니다.")

LightGBM 설치됨
버전: 4.7.0


### 6-2. 60초 LightGBM 모델 학습

In [38]:
lgb_60s_models = {}

for component in component_order:

    y_train = profile[
        profile["cycle_id"].isin(train_ids)
    ][component]

    model = LGBMClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    model.fit(X_train, y_train)

    lgb_60s_models[component] = model

    print(component, "60초 LightGBM 학습 완료")

cooler 60초 LightGBM 학습 완료
valve 60초 LightGBM 학습 완료
pump 60초 LightGBM 학습 완료
accumulator 60초 LightGBM 학습 완료


### 6-3. LightGBM Accuracy / Macro F1 평가

In [39]:
lgb_results = []

for component in component_order:

    y_val = profile[
        profile["cycle_id"].isin(val_ids)
    ][component]

    pred = lgb_60s_models[component].predict(X_val)

    lgb_results.append({
        "component": component,
        "accuracy": accuracy_score(y_val, pred),
        "macro_f1": f1_score(
            y_val,
            pred,
            average="macro"
        )
    })

lgb_results = pd.DataFrame(lgb_results)

display(lgb_results)

,component,accuracy,macro_f1
0,cooler,1.000000,1.000000
1,valve,0.561934,0.179884
2,pump,0.830816,0.679533
3,accumulator,0.365559,0.178466


### 6-4. RandomForest / LightGBM 성능 비교

In [40]:
rf_compare = results.copy()
rf_compare["model"] = "RandomForest"

lgb_compare = lgb_results.copy()
lgb_compare["model"] = "LightGBM"

model_comparison = pd.concat(
    [rf_compare, lgb_compare],
    ignore_index=True
)

model_comparison = model_comparison[
    ["model", "component", "accuracy", "macro_f1"]
]

display(model_comparison)

,model,component,accuracy,macro_f1
0,RandomForest,cooler,1.000000,1.000000
1,RandomForest,valve,0.561934,0.179884
2,RandomForest,pump,0.794562,0.715191
3,RandomForest,accumulator,0.365559,0.178466
4,LightGBM,cooler,1.000000,1.000000
5,LightGBM,valve,0.561934,0.179884
6,LightGBM,pump,0.830816,0.679533
7,LightGBM,accumulator,0.365559,0.178466


In [41]:
model_f1_table = model_comparison.pivot(
    index="component",
    columns="model",
    values="macro_f1"
)

model_f1_table = model_f1_table.loc[
    component_order,
    ["RandomForest", "LightGBM"]
]

display(model_f1_table)

model,RandomForest,LightGBM
component,,
cooler,1.000000,1.000000
valve,0.179884,0.179884
pump,0.715191,0.679533
accumulator,0.178466,0.178466


In [42]:
best_model_results = []

for component in component_order:

    rf_score = model_f1_table.loc[component, "RandomForest"]
    lgb_score = model_f1_table.loc[component, "LightGBM"]

    if rf_score > lgb_score:
        best_model = "RandomForest"
        best_score = rf_score

    elif lgb_score > rf_score:
        best_model = "LightGBM"
        best_score = lgb_score

    else:
        best_model = "Same"
        best_score = rf_score

    best_model_results.append({
        "component": component,
        "best_model": best_model,
        "best_macro_f1": best_score
    })

best_model_results = pd.DataFrame(best_model_results)

display(best_model_results)

,component,best_model,best_macro_f1
0,cooler,Same,1.000000
1,valve,Same,0.179884
2,pump,RandomForest,0.715191
3,accumulator,Same,0.178466


## 7. 성능 개선 및 검증 방식 비교

### 7-1. class_weight 적용 RandomForest 비교

#### class_weight 비교 결과
- `class_weight="balanced"` 적용 후 밸브와 축압기의 Macro F1은 개선되지 않았다.
- 펌프는 오히려 Macro F1이 감소하였다.
- 따라서 현재 성능 저하의 주요 원인이 단순 클래스 불균형이라고 보기는 어렵다.
- 이후 분할 방식 및 특징의 영향을 추가로 확인한다.

In [43]:
# 클래스 개수 차이의 영향을 줄이기 위해 class_weight="balanced" 모델을 추가로 학습

rf_balanced_models = {}

for component in component_order:

    y_train = profile[
        profile["cycle_id"].isin(train_ids)
    ][component]

    model = RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    rf_balanced_models[component] = model

    print(component, "Balanced RandomForest 학습 완료")

cooler Balanced RandomForest 학습 완료
valve Balanced RandomForest 학습 완료
pump Balanced RandomForest 학습 완료
accumulator Balanced RandomForest 학습 완료


In [44]:
balanced_results = []

for component in component_order:

    y_val = profile[
        profile["cycle_id"].isin(val_ids)
    ][component]

    pred = rf_balanced_models[component].predict(X_val)

    balanced_results.append({
        "component": component,
        "accuracy": accuracy_score(y_val, pred),
        "macro_f1": f1_score(
            y_val,
            pred,
            average="macro"
        )
    })

balanced_results = pd.DataFrame(balanced_results)

display(balanced_results)

,component,accuracy,macro_f1
0,cooler,1.000000,1.000000
1,valve,0.561934,0.179884
2,pump,0.764350,0.669498
3,accumulator,0.365559,0.178466


In [45]:
rf_weight_compare = results[
    ["component", "accuracy", "macro_f1"]
].copy()

rf_weight_compare = rf_weight_compare.rename(
    columns={
        "accuracy": "basic_accuracy",
        "macro_f1": "basic_macro_f1"
    }
)

balanced_compare = balanced_results.rename(
    columns={
        "accuracy": "balanced_accuracy",
        "macro_f1": "balanced_macro_f1"
    }
)

rf_weight_compare = rf_weight_compare.merge(
    balanced_compare,
    on="component"
)

rf_weight_compare = rf_weight_compare.set_index("component").loc[
    component_order
].reset_index()

display(rf_weight_compare)

,component,basic_accuracy,basic_macro_f1,balanced_accuracy,balanced_macro_f1
0,cooler,1.000000,1.000000,1.000000,1.000000
1,valve,0.561934,0.179884,0.561934,0.179884
2,pump,0.794562,0.715191,0.764350,0.669498
3,accumulator,0.365559,0.178466,0.365559,0.178466


### 7-2. 분할 방법 변경 비교 - Stratified 분할

#### Stratified는 Train과 Validation을 나눌 때 정답 클래스의 비율을 비슷하게 유지해서 나누는 데이터 분할 방법
#### 각 상태 클래스 비율이 Train과 Validation에 비슷하게 유지되도록 Stratified 방식으로 분할

In [46]:
# 기존 Train + Validation 범위만 사용
dev_data = features_60s[
    features_60s["cycle_id"].between(1, 1874)
].copy()

X_dev = dev_data.drop(columns=["cycle_id"])

print("개발용 전체 데이터:", X_dev.shape)

개발용 전체 데이터: (1874, 119)


In [47]:
# 부품별 Stratified RandomForest 학습

stratified_results = []
rf_stratified_models = {}

for component in component_order:

    # cycle_id에 맞는 정답
    y_dev = profile[
        profile["cycle_id"].between(1, 1874)
    ][component]

    # 각 클래스 비율을 유지하면서 Train / Validation 분할
    X_train_s, X_val_s, y_train_s, y_val_s = train_test_split(
        X_dev,
        y_dev,
        test_size=331,
        random_state=42,
        stratify=y_dev
    )

    model = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_s, y_train_s)

    pred = model.predict(X_val_s)

    rf_stratified_models[component] = model

    stratified_results.append({
        "component": component,
        "accuracy": accuracy_score(y_val_s, pred),
        "macro_f1": f1_score(
            y_val_s,
            pred,
            average="macro"
        )
    })

stratified_results = pd.DataFrame(stratified_results)

display(stratified_results)

,component,accuracy,macro_f1
0,cooler,1.000000,1.000000
1,valve,0.996979,0.996869
2,pump,0.996979,0.995305
3,accumulator,0.972810,0.967302


In [48]:
# 기존 시간순 분할과 비교

split_compare = results[
    ["component", "accuracy", "macro_f1"]
].copy()

split_compare = split_compare.rename(
    columns={
        "accuracy": "time_accuracy",
        "macro_f1": "time_macro_f1"
    }
)

strat_compare = stratified_results.rename(
    columns={
        "accuracy": "stratified_accuracy",
        "macro_f1": "stratified_macro_f1"
    }
)

split_compare = split_compare.merge(
    strat_compare,
    on="component"
)

split_compare = split_compare.set_index("component").loc[
    component_order
].reset_index()

display(split_compare)

,component,time_accuracy,time_macro_f1,stratified_accuracy,stratified_macro_f1
0,cooler,1.000000,1.000000,1.000000,1.000000
1,valve,0.561934,0.179884,0.996979,0.996869
2,pump,0.794562,0.715191,0.996979,0.995305
3,accumulator,0.365559,0.178466,0.972810,0.967302


### 7-3. 시간순 분할 추가 검증 - Rolling 방식

In [49]:
# 검증 구간 설정

rolling_folds = [
    {
        "fold": "1차",
        "train_start": 1,
        "train_end": 1000,
        "val_start": 1001,
        "val_end": 1200
    },
    {
        "fold": "2차",
        "train_start": 1,
        "train_end": 1200,
        "val_start": 1201,
        "val_end": 1400
    },
    {
        "fold": "3차",
        "train_start": 1,
        "train_end": 1400,
        "val_start": 1401,
        "val_end": 1600
    },
    {
        "fold": "4차",
        "train_start": 1,
        "train_end": 1600,
        "val_start": 1601,
        "val_end": 1874
    }
]

for fold in rolling_folds:
    print(
        fold["fold"],
        "Train:",
        fold["train_start"], "~", fold["train_end"],
        "| Validation:",
        fold["val_start"], "~", fold["val_end"]
    )

1차 Train: 1 ~ 1000 | Validation: 1001 ~ 1200
2차 Train: 1 ~ 1200 | Validation: 1201 ~ 1400
3차 Train: 1 ~ 1400 | Validation: 1401 ~ 1600
4차 Train: 1 ~ 1600 | Validation: 1601 ~ 1874


In [50]:
# 과거 데이터로 학습하고 이후 시간대 데이터로 검증하여 실제 시간 흐름에서도 성능이 유지되는지 확인

rolling_results = []

for fold in rolling_folds:

    # 시간순 Train / Validation 특징 데이터
    train_fold = features_60s[
        features_60s["cycle_id"].between(
            fold["train_start"],
            fold["train_end"]
        )
    ]

    val_fold = features_60s[
        features_60s["cycle_id"].between(
            fold["val_start"],
            fold["val_end"]
        )
    ]

    # cycle_id 제외
    X_train_fold = train_fold.drop(columns=["cycle_id"])
    X_val_fold = val_fold.drop(columns=["cycle_id"])

    for component in component_order:

        # cycle_id 순서에 맞는 정답
        y_train_fold = profile[
            profile["cycle_id"].between(
                fold["train_start"],
                fold["train_end"]
            )
        ][component]

        y_val_fold = profile[
            profile["cycle_id"].between(
                fold["val_start"],
                fold["val_end"]
            )
        ][component]

        model = RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        )

        model.fit(X_train_fold, y_train_fold)

        pred = model.predict(X_val_fold)

        rolling_results.append({
            "fold": fold["fold"],
            "component": component,
            "train_range": f'{fold["train_start"]}~{fold["train_end"]}',
            "val_range": f'{fold["val_start"]}~{fold["val_end"]}',
            "accuracy": accuracy_score(y_val_fold, pred),
            "macro_f1": f1_score(
                y_val_fold,
                pred,
                average="macro"
            )
        })

rolling_results = pd.DataFrame(rolling_results)

display(rolling_results)

,fold,component,train_range,val_range,accuracy,macro_f1
0,1차,cooler,1~1000,1001~1200,1.000000,1.000000
1,1차,valve,1~1000,1001~1200,0.515000,0.440412
2,1차,pump,1~1000,1001~1200,1.000000,1.000000
3,1차,accumulator,1~1000,1001~1200,0.325000,0.163522
4,2차,cooler,1~1200,1201~1400,1.000000,1.000000
5,2차,valve,1~1200,1201~1400,0.945000,0.940072
6,2차,pump,1~1200,1201~1400,1.000000,1.000000
7,2차,accumulator,1~1200,1201~1400,0.040000,0.037037
8,3차,cooler,1~1400,1401~1600,0.320000,0.242424
9,3차,valve,1~1400,1401~1600,0.995000,0.984330


In [51]:
# Macro F1 한눈에 비교

rolling_f1_table = rolling_results.pivot(
    index="component",
    columns="fold",
    values="macro_f1"
)

rolling_f1_table = rolling_f1_table.loc[
    component_order,
    ["1차", "2차", "3차", "4차"]
]

# 평균 성능 추가
rolling_f1_table["평균"] = rolling_f1_table.mean(axis=1)

display(rolling_f1_table)

fold,1차,2차,3차,4차,평균
component,,,,,
cooler,1.000000,1.000000,0.242424,1.000000,0.810606
valve,0.440412,0.940072,0.984330,0.160050,0.631216
pump,1.000000,1.000000,1.000000,0.852578,0.963145
accumulator,0.163522,0.037037,0.012195,0.126233,0.084747


### 7-4. 시간별 특징 데이터에 Stratified 분할 적용

In [52]:
# 시간별 특징 데이터 준비

feature_sets = {
    "10s": features_10s,
    "20s": features_20s,
    "30s": features_30s,
    "60s": features_60s
}

for window, df in feature_sets.items():
    print(window, df.shape)

10s (2205, 120)
20s (2205, 120)
30s (2205, 120)
60s (2205, 120)


In [53]:
# 같은 Stratified 분할로 4개 시간 비교
stratified_window_results = []

# 최종 Test를 제외한 개발 데이터 cycle_id
dev_ids = profile[
    profile["cycle_id"].between(1, 1874)
]["cycle_id"]

for component in component_order:

    # 해당 부품 정답
    y_dev = profile[
        profile["cycle_id"].between(1, 1874)
    ][["cycle_id", component]].copy()

    # 먼저 cycle_id 자체를 Stratified 방식으로 분할
    train_cycle_ids, val_cycle_ids = train_test_split(
        y_dev["cycle_id"],
        test_size=331,
        random_state=42,
        stratify=y_dev[component]
    )

    train_cycle_ids = set(train_cycle_ids)
    val_cycle_ids = set(val_cycle_ids)

    print("=" * 50)
    print(component)
    print("Train:", len(train_cycle_ids))
    print("Validation:", len(val_cycle_ids))

    # 같은 cycle_id 분할을 10/20/30/60초 모두에 사용
    for window, df in feature_sets.items():

        train_df = df[
            df["cycle_id"].isin(train_cycle_ids)
        ].sort_values("cycle_id")

        val_df = df[
            df["cycle_id"].isin(val_cycle_ids)
        ].sort_values("cycle_id")

        X_train_w = train_df.drop(columns=["cycle_id"])
        X_val_w = val_df.drop(columns=["cycle_id"])

        # 특징 데이터 cycle_id 순서에 맞춰 정답 가져오기
        y_train_w = (
            profile[
                profile["cycle_id"].isin(train_cycle_ids)
            ]
            .sort_values("cycle_id")[component]
        )

        y_val_w = (
            profile[
                profile["cycle_id"].isin(val_cycle_ids)
            ]
            .sort_values("cycle_id")[component]
        )

        model = RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        )

        model.fit(X_train_w, y_train_w)

        pred = model.predict(X_val_w)

        stratified_window_results.append({
            "component": component,
            "window": window,
            "accuracy": accuracy_score(y_val_w, pred),
            "macro_f1": f1_score(
                y_val_w,
                pred,
                average="macro"
            )
        })

        print(window, "완료")

cooler
Train: 1543
Validation: 331
10s 완료
20s 완료
30s 완료
60s 완료
valve
Train: 1543
Validation: 331
10s 완료
20s 완료
30s 완료
60s 완료
pump
Train: 1543
Validation: 331
10s 완료
20s 완료
30s 완료
60s 완료
accumulator
Train: 1543
Validation: 331
10s 완료
20s 완료
30s 완료
60s 완료


In [54]:
# Macro F1 비교표

stratified_window_results = pd.DataFrame(
    stratified_window_results
)

stratified_window_f1 = stratified_window_results.pivot(
    index="component",
    columns="window",
    values="macro_f1"
)

stratified_window_f1 = stratified_window_f1.loc[
    component_order,
    ["10s", "20s", "30s", "60s"]
]

display(stratified_window_f1)

window,10s,20s,30s,60s
component,,,,
cooler,1.000000,1.000000,1.000000,1.000000
valve,1.000000,0.993694,0.996869,0.993780
pump,0.995303,0.993551,0.996789,0.990610
accumulator,0.983404,0.980269,0.978853,0.971162


In [55]:
# 각 부품 최고 시간 자동 확인

stratified_best_window = []

for component in component_order:

    scores = stratified_window_f1.loc[component]

    stratified_best_window.append({
        "component": component,
        "best_window": scores.idxmax(),
        "best_macro_f1": scores.max()
    })

stratified_best_window = pd.DataFrame(
    stratified_best_window
)

display(stratified_best_window)

,component,best_window,best_macro_f1
0,cooler,10s,1.000000
1,valve,10s,1.000000
2,pump,30s,0.996789
3,accumulator,10s,0.983404
